In [2]:
import numpy as np

n = 200          # individuals
m = 100           # SNPs
p = (m*(m-1))/2
maf_low = 0.05
maf_high = 0.5
seed = 0

rng = np.random.default_rng(seed)
# ---- genotypes ----
X = rng.binomial(2, 0.2, size=(n, m))          # 0 / 1 / 2

# ---- standardization ----
Z = (X - X.mean(axis=0)) / X.std(axis=0)     # empirical, per-column
D = Z*Z

# ---- epistasis design matrix ----
j, k = np.triu_indices(m, k=1)               
H = Z[:, j] * Z[:, k]                       


H_std = (H - H.mean(axis=0)) / H.std(axis=0)

W = H@H.T / p
W_std = H_std@H_std.T / p

u = rng.normal(0, 1, size=n)
s_u = np.sum(u)

In [138]:
#exact Wu
Wu_exact = W@u
Wstdu_exact = W_std@u

#O(mn2) method

M = Z.T @ np.diag(u) @ Z
Wu_operator = ((Z*(Z@M))@np.ones(m) - D@(D.T @ u))/(2*p)

diff = Wu_operator - Wu_exact
rel = np.linalg.norm(diff) / np.linalg.norm(Wu_exact)
print(f'Wu_exact = Wu_operator? {rel < 1e-8}')

#std
mu = Z.T @ Z / n                           
m2 = D.T @ D / n                        
sigma2 = m2 - mu ** 2
np.fill_diagonal(sigma2, 1.0)              # placeholder to avoid division by zero
B = 1.0 / sigma2
np.fill_diagonal(B, 0.0)                   # pairs only, j != k
R = B * mu
v_R = (Z * (Z @ R)) @ np.ones(m)
s_T = v_R.mean()

dB = B - 1.0                               
np.fill_diagonal(dB, 0.0)                  
delta = (Z * (Z @ (dB * M))) @ np.ones(m)  
Wu_operator_std = ((Z * (Z @ (B * M))) @ np.ones(m) - s_u * v_R + (s_u * s_T - u @ v_R)) / (2 * p)

diff = Wstdu_exact - Wu_operator_std
rel = np.linalg.norm(diff) / np.linalg.norm(Wstdu_exact)
print(f'Wstdu_exact = Wu_operator_std? {rel < 1e-8}')

Wu_exact = Wu_operator? True
Wstdu_exact = Wu_operator_std? True


In [86]:
#compare stochastic Wu with unstd Wu
import numpy as np

n = 500
m = 200
p = m * (m - 1) / 2
Nmc = 100
reps = 5000

rng = np.random.default_rng()        

X = rng.binomial(2, 0.2, size=(n, m))
Z = (X - X.mean(axis=0)) / X.std(axis=0)
D = Z * Z
u = rng.normal(0, 1, size=n)

# ---- exact reference, fixed throughout ----
j, k = np.triu_indices(m, k=1)
H = Z[:, j] * Z[:, k]
Wu_exact = H @ (H.T @ u) / p
Dterm = D @ (D.T @ u)

# ---- average many stochastic estimates ----
acc = np.zeros(n)
for r in range(1, reps + 1):
    V = rng.integers(0, 2, size=(n, Nmc)) * 2.0 - 1.0
    T1 = Z @ (Z.T @ V)
    T1 = u[:, None] * T1
    T1 = Z @ (Z.T @ T1)
    acc += ((T1 * V).mean(axis=1) - Dterm) / (2 * p)

    if r in (1, 10, 100, 500, 2000,5000):
        avg = acc / r
        rel_err = (avg - Wu_exact).mean()
        rel_L2 = np.linalg.norm(avg - Wu_exact) / np.linalg.norm(Wu_exact)
        print(f"Nmc = {r*Nmc:7d}  mean relative err = {rel_err:+.3e}  rel L2 norm err = {rel_L2:.4f}")

print("The front 5 element of exact Wu and the average of Stochastic Wu:")
print("Exact  ", np.round(Wu_exact[:5], 4))
print("Stochasitic", np.round(acc[:5] / reps, 4))

Nmc =     100  mean relative err = -5.013e-04  rel L2 norm err = 0.3295
Nmc =    1000  mean relative err = +1.661e-02  rel L2 norm err = 0.1103
Nmc =   10000  mean relative err = +1.054e-03  rel L2 norm err = 0.0320
Nmc =   50000  mean relative err = -5.631e-04  rel L2 norm err = 0.0147
Nmc =  200000  mean relative err = -8.482e-04  rel L2 norm err = 0.0076
Nmc =  500000  mean relative err = -4.522e-04  rel L2 norm err = 0.0046
The front 5 element of exact Wu and the average of Stochastic Wu:
Exact   [-0.7696 -2.3769  0.3389  0.5216 -0.7693]
Stochasitic [-0.7743 -2.3879  0.3477  0.5161 -0.772 ]


In [74]:
# variance of the stochastic Wu estimator as a function of Nmc
import numpy as np

n = 500
m = 200
p = m * (m - 1) / 2
reps = 100
Nmc_list = [50, 100, 200, 400, 800, 1600]

rng = np.random.default_rng()

X = rng.binomial(2, 0.2, size=(n, m))
Z = (X - X.mean(axis=0)) / X.std(axis=0)
D = Z * Z
u = rng.normal(0, 1, size=n)
Dterm = D @ (D.T @ u)

# ---- exact reference, for the relative scale only ----
j, k = np.triu_indices(m, k=1)
H = Z[:, j] * Z[:, k]
Wu_exact = H @ (H.T @ u) / p
scale2 = (Wu_exact ** 2).mean()

print(f"{'Nmc':>6}{'avg var':>12}")
for Nmc in Nmc_list:
    s1 = np.zeros(n)
    s2 = np.zeros(n)
    for _ in range(reps):
        V = rng.integers(0, 2, size=(n, Nmc)) * 2.0 - 1.0
        T1 = Z @ (Z.T @ V)
        T1 = u[:, None] * T1
        T1 = Z @ (Z.T @ T1)
        e = ((T1 * V).mean(axis=1) - Dterm) / (2 * p)
        s1 += e
        s2 += e ** 2

    mean = s1 / reps
    var = np.maximum(s2 / reps - mean ** 2, 0)
    avg_var = var.mean()

    print(f"{Nmc:6d}{avg_var:12.5f}")

   Nmc     avg var
    50     0.23846
   100     0.12194
   200     0.05964
   400     0.03011
   800     0.01476
  1600     0.00749


In [79]:
# variance of the stochastic Wu estimator as a function of n
import numpy as np

m = 500
p = m * (m - 1) / 2
Nmc = 100
reps = 100
n_list = [125, 250, 500, 1000, 2000, 4000, 8000]

rng = np.random.default_rng()

print(f"{'n':>6}{'avg var':>12}")
for n in n_list:
    X = rng.binomial(2, 0.2, size=(n, m))
    Z = (X - X.mean(axis=0)) / X.std(axis=0)
    D = Z * Z
    u = rng.normal(0, 1, size=n)
    Dterm = D @ (D.T @ u)

    s1 = np.zeros(n)
    s2 = np.zeros(n)
    for _ in range(reps):
        V = rng.integers(0, 2, size=(n, Nmc)) * 2.0 - 1.0
        T1 = Z @ (Z.T @ V)
        T1 = u[:, None] * T1
        T1 = Z @ (Z.T @ T1)
        e = ((T1 * V).mean(axis=1) - Dterm) / (2 * p)
        s1 += e
        s2 += e ** 2
    mean = s1 / reps
    var = np.maximum(s2 / reps - mean ** 2, 0)
    avg_var = var.mean()
    print(f"{n:6d}{avg_var:12.5f}")

     n     avg var
   125     0.00857
   250     0.01190
   500     0.02774
  1000     0.07921
  2000     0.24280
  4000     0.81178
  8000     2.86877


In [80]:
# variance of the stochastic Wu estimator as a function of m
import numpy as np

n = 500
Nmc = 100
reps = 100
m_list = [50, 100, 200, 400, 800]

rng = np.random.default_rng()

print(f"{'m':>6}{'avg var':>12}")
for m in m_list:
    p = m * (m - 1) / 2
    X = rng.binomial(2, 0.2, size=(n, m))
    Z = (X - X.mean(axis=0)) / X.std(axis=0)
    D = Z * Z
    u = rng.normal(0, 1, size=n)
    Dterm = D @ (D.T @ u)

    s1 = np.zeros(n)
    s2 = np.zeros(n)
    for _ in range(reps):
        V = rng.integers(0, 2, size=(n, Nmc)) * 2.0 - 1.0
        T1 = Z @ (Z.T @ V)
        T1 = u[:, None] * T1
        T1 = Z @ (Z.T @ T1)
        e = ((T1 * V).mean(axis=1) - Dterm) / (2 * p)
        s1 += e
        s2 += e ** 2
    mean = s1 / reps
    var = np.maximum(s2 / reps - mean ** 2, 0)
    avg_var = var.mean()
    print(f"{m:6d}{avg_var:12.5f}")

     m     avg var
    50     1.37330
   100     0.37100
   200     0.10844
   400     0.04120
   800     0.01719


In [83]:
# variance of the stochastic Wu estimator as n and m grow proportionally
import numpy as np

ratio = 1                      
Nmc = 100
reps = 100
m_list = [50, 100, 200, 400, 800]

rng = np.random.default_rng()

print(f"{'n':>6}{'m':>6}{'avg var':>12}")
for m in m_list:
    n = int(ratio * m)
    p = m * (m - 1) / 2
    X = rng.binomial(2, 0.2, size=(n, m))
    Z = (X - X.mean(axis=0)) / X.std(axis=0)
    D = Z * Z
    u = rng.normal(0, 1, size=n)
    Dterm = D @ (D.T @ u)

    s1 = np.zeros(n)
    s2 = np.zeros(n)
    for _ in range(reps):
        V = rng.integers(0, 2, size=(n, Nmc)) * 2.0 - 1.0
        T1 = Z @ (Z.T @ V)
        T1 = u[:, None] * T1
        T1 = Z @ (Z.T @ T1)
        e = ((T1 * V).mean(axis=1) - Dterm) / (2 * p)
        s1 += e
        s2 += e ** 2
    mean = s1 / reps
    var = np.maximum(s2 / reps - mean ** 2, 0)
    avg_var = var.mean()
    print(f"{n:6d}{m:6d}{avg_var:12.5f}")

     n     m     avg var
    50    50     0.03171
   100   100     0.02837
   200   200     0.03198
   400   400     0.03100
   800   800     0.02983
